# Agentic RAG: set up, langsmith tracing

**Placement:** After the introductory FAISS RAG notebook and **before** the Tools & APIs lecture.

**Learning goals:** (1) build an agent that *decides whether to retrieve* company documents, (2) extend that agent with a simple Python tool, and (3) observe which nodes run.

We reuse the fictional **AcmeOrbit** internal policies. **FAISS is our vector store.** The second example adds a simple **calculator tool** (no REST API or server yet). Both examples use explicit **LangGraph nodes and edges**. Run cells top to bottom. Requires an OpenAI API key and API access.

## 0. Install and configure

In [ ]:
%pip install -q langgraph langchain-openai langchain-community langchain-core faiss-cpu python-dotenv

In [ ]:
import os, getpass
from colorama import Fore, Style
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition

load_dotenv(override=True)
model = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")
else:
    print(Style.BRIGHT + Fore.GREEN + "Using existing OpenAI API key." + Style.RESET_ALL)
    
llm = ChatOpenAI(model=model, temperature=0)



## 1. Create a tiny FAISS vector store

In [ ]:
documents = [
    Document(page_content="AcmeOrbit IT policy: Employees must return a borrowed laptop within 14 calendar days after the loan ends.", metadata={"source": "IT-LAPTOP-01"}),
    Document(page_content="AcmeOrbit travel policy: Expense claims must be submitted within 30 calendar days after travel.", metadata={"source": "TRAVEL-02"}),
    Document(page_content="AcmeOrbit security policy: Employees must use multifactor authentication for company accounts.", metadata={"source": "SECURITY-03"}),
]
vectorstore = FAISS.from_documents(documents, OpenAIEmbeddings(model="text-embedding-3-small"))
print("FAISS documents:", vectorstore.index.ntotal)

def retrieve_context(query: str, k: int = 1) -> str:
    hits = vectorstore.similarity_search(query, k=k)
    return "\n\n".join(f"Source: {d.metadata['source']}\n{d.page_content}" for d in hits)


## 2. Example A: simple **agentic RAG**

- In ordinary RAG, retrieval happens **every time**. 
- In agentic RAG, the model decices whether retrieval is needed:

`START → decide → (retrieve → answer | answer) → END`

The `decide` node returns a structured `needs_retrieval` boolean. 

In [ ]:
from typing import TypedDict
from pydantic import BaseModel, Field

class RouteDecision(BaseModel):
    needs_retrieval: bool = Field(description="True if the question requires AcmeOrbit internal policy facts")

class RAGState(TypedDict):
    question: str
    needs_retrieval: bool
    context: str
    answer: str

router = llm.with_structured_output(RouteDecision)

def decide_node(state: RAGState):
    choice = router.invoke([
        ("system", "Set needs_retrieval=true for questions about AcmeOrbit's internal policies. Set false for general knowledge or greetings."),
        ("human", state["question"]),
    ])
    print("NODE decide → needs_retrieval:", choice.needs_retrieval)
    return {"needs_retrieval": choice.needs_retrieval}

def route_after_decide(state: RAGState):
    return "retrieve" if state["needs_retrieval"] else "answer"

def retrieve_node(state: RAGState):
    context = retrieve_context(state["question"])
    print("NODE retrieve →", context.replace("\n", " | "))
    return {"context": context}

def answer_node(state: RAGState):
    print("NODE answer")
    if state["needs_retrieval"]:
        system = ("Answer using ONLY the supplied company policy context. "
                  "If the context does not explicitly answer the question, say 'I don't know'. Cite the source ID.")
        human = f"Context:\n{state.get('context', '')}\n\nQuestion: {state['question']}"
    else:
        system = "Answer this general question concisely. You have no company policy context."
        human = state["question"]
    response = llm.invoke([("system", system), ("human", human)])
    return {"answer": response.content}

# build the agentic RAG graph
builder_a = StateGraph(RAGState)
builder_a.add_node("decide", decide_node)
builder_a.add_node("retrieve", retrieve_node)
builder_a.add_node("answer", answer_node)
builder_a.add_edge(START, "decide")
builder_a.add_conditional_edges("decide", route_after_decide, {"retrieve": "retrieve", "answer": "answer"})
builder_a.add_edge("retrieve", "answer")
builder_a.add_edge("answer", END)
agentic_rag = builder_a.compile()

png_data = agentic_rag.get_graph().draw_mermaid_png()
with open("agentic_rag_graph.png", "wb") as f:
    f.write(png_data)



Run the agentic flow

In [ ]:
questions = [
    "How many days do AcmeOrbit employees have to return a borrowed laptop?",
    "What is 2 + 2?",
]

for question in questions:
    result = agentic_rag.invoke(
        {
            "question": question,
            "needs_retrieval": False,
            "context": "",
            "answer": "",
        },
        config={"recursion_limit": 8},
    )

    print("\nQUESTION:", question)
    print("ANSWER:", result["answer"])

**Observe:** The first question should run `decide → retrieve → answer`; the general question should skip retrieval. Routing is model-driven, so inspect the printed node trace rather than assuming every run is identical.

**Limitation:** FAISS returns nearest matches even when no document answers a question. The answer prompt instructs the model to abstain, but production systems also add relevance checks.

## 3. Example B: **agentic RAG + a tool**

Now the agent can choose between **two tools**:

- `search_company_policy`: looks up internal company policy using FAISS.

The LLM decides which tool(s) to call. LangGraph's `ToolNode` executes them. The graph is:

`START → agent → (tools → agent | END)`

This is a bridge to the next lecture, where tools will call REST APIs.

In [ ]:
from datetime import date, timedelta

@tool
def search_company_policy(question: str) -> str:
    """Search AcmeOrbit internal policies. Use for any question about company rules or deadlines."""
    print("TOOL search_company_policy called with:", question)
    return retrieve_context(question)

tools = [search_company_policy]
llm_with_tools = llm.bind_tools(tools)

SYSTEM_B = ("You are an AcmeOrbit helpdesk assistant. "
            "For company policy questions, you MUST call search_company_policy before answering. "
            "Never invent company policies. Only state company-policy facts present in tool results; "
            "if absent, say 'I don't know'. Cite policy source IDs. "
            "Answer clearly and concisely.")

def agent_node(state: MessagesState):
    print("NODE agent")
    response = llm_with_tools.invoke([SystemMessage(content=SYSTEM_B)] + state["messages"])
    return {"messages": [response]}

# build the agentic RAG graph with tool usage
